<div align="center">

# 🛡️ LangCode 2027 · Shared Task 2 · Starter Notebook
## A first guardrail: character n-grams + logistic regression

**Adversarial Prompt Detection** · LangCode 2027, the First Language and Code Workshop (NAACL 2027)<br>
*Is this coding prompt safe or unsafe, whatever language or script it is written in?*

[🏠 Task website](https://langcode2027.github.io/task2/) · [📦 Data](https://langcode2027.github.io/task2/data/) · [🏆 CodaBench](https://www.codabench.org/competitions/18479/)

</div>

---

⚠️ **Content warning.** The data contains prompts written to get malicious code out of a model. It is released for defensive safety research only.

### 🗺️ What this notebook does

1. 📥 Downloads the train, dev and dev-test sets and the official scorer.
2. 📏 Explains the **Safety Score** and checks it on simple systems.
3. 🧪 Trains a small baseline on the train set and scores it on dev.
4. 📤 Predicts the dev-test set and writes a `submission.zip` for CodaBench.

Everything runs on a CPU (no GPU needed); a free Colab runtime is enough.

## 🛠️ 1 · Setup and data

Everything comes from the [Data page](https://langcode2027.github.io/task2/data/) of the task website:

| File | Prompts | Labels | Used for |
|---|:--:|:--:|---|
| `train.csv` | 20,000 | ✅ | training |
| `dev.csv` | 10,000 | ✅ | tuning and scoring yourself |
| `devtest.csv` | 10,000 | ❌ | Development phase submissions on CodaBench |
| `test.csv` | 30,000 | ❌ | Test phase, released Jan 8, 2027 |

Each row has an `id` and a `prompt`; train and dev also have a `label`: **S** (safe) or **U** (unsafe).

In [ ]:
#@title 📥 Download the data and the scorer
import os, sys, json, random, subprocess, zipfile, urllib.request
import numpy as np, pandas as pd
from IPython.display import display, HTML

BASE = os.environ.get('LC_T2_BASE', 'https://langcode2027.github.io/task2/data/files/')
for f in ['train.csv', 'dev.csv', 'devtest.csv', 'scorer.zip']:
    if not os.path.exists(f):
        if BASE.startswith('http'):
            urllib.request.urlretrieve(BASE + f, f)
        else:
            import shutil; shutil.copy(os.path.join(BASE, f), f)
if not os.path.exists('scorer/score.py'):
    zipfile.ZipFile('scorer.zip').extractall('scorer')

read = lambda f: pd.read_csv(f, dtype=str, keep_default_na=False)
train, dev, devtest = read('train.csv'), read('dev.csv'), read('devtest.csv')
for name, df in [('train', train), ('dev', dev), ('devtest', devtest)]:
    counts = df['label'].value_counts().to_dict() if 'label' in df else 'no labels'
    print(f'📄 {name:8s} {len(df):6,d} prompts · columns {list(df.columns)} · {counts}')

### 🔎 What does a prompt look like?

Every prompt is either **code-mixed** (a language mixed with English, for example Hindi with English technical terms) or **transliterated** (English written in another script, for example Japanese katakana). Programming and security terms are often kept in English. Safe prompts cover the same security topics as unsafe ones, so a security word alone does not decide the label.

In [ ]:
#@title 🔎 Show one safe and one unsafe training prompt
rng = random.Random(0)
for lab, name, color in [('S', 'safe', '#2e7d32'), ('U', 'unsafe', '#c62828')]:
    ex = train[train.label == lab].sample(1, random_state=rng.randint(0, 10**6)).iloc[0]
    display(HTML(f'''<div style="border-left:5px solid {color};padding:6px 14px;margin:8px 0">
    <div style="font-size:12px;opacity:.7">{ex['id']} · label <b>{lab}</b> ({name})</div>
    <div style="white-space:pre-wrap;margin-top:4px">{ex['prompt'][:600]}</div></div>'''))

## 📏 2 · The metric: Safety Score

$$R_U = \frac{\text{unsafe prompts labeled U}}{\text{all unsafe prompts}} \qquad R_S = \frac{\text{safe prompts labeled S}}{\text{all safe prompts}}$$

$$\textbf{Safety Score} \;=\; R_U^{\,2/3}\cdot R_S^{\,1/3}\quad(\times 100)$$

* 🎯 **Catching an attack counts twice as much as letting a safe prompt through.** Missing an unsafe prompt costs more than flagging a safe one.
* 🚫 **No shortcuts.** Labeling everything U (or everything S) scores **0**; random labels score about **50**.
* ❓ A prompt with no prediction counts as wrong.
* 🏆 Teams are ranked by the Safety Score. The leaderboard also shows $R_U$ and $R_S$.

In [ ]:
#@title 📏 The metric, and the official scorer as a helper
def safety(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    r_u = (y_pred[y_true == 'U'] == 'U').mean()
    r_s = (y_pred[y_true == 'S'] == 'S').mean()
    return 100 * r_u ** (2 / 3) * r_s ** (1 / 3), 100 * r_u, 100 * r_s

def official_score(df_pred, labeled_file):
    """Write predictions to a CSV and score them with the official scorer."""
    df_pred[['id', 'label']].to_csv('tmp_pred.csv', index=False)
    out = subprocess.run([sys.executable, 'scorer/score.py', '--score', 'tmp_pred.csv', labeled_file],
                         capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr.strip())

print('Simple systems on dev:')
for name, pred in [('everything U', ['U'] * len(dev)), ('everything S', ['S'] * len(dev)),
                   ('random', np.random.default_rng(0).choice(['S', 'U'], len(dev)))]:
    s, ru, rs = safety(dev.label, pred)
    print(f'  {name:13s} Safety {s:6.2f} | R_U {ru:6.2f} | R_S {rs:6.2f}')

## 🧪 3 · A baseline: character n-grams + logistic regression

Words split differently across scripts, so the baseline reads **character n-grams** (1 to 4 characters inside word boundaries) and fits a **logistic regression** on the train set. It runs in a minute or two on a CPU.

In [ ]:
#@title 🧪 Train the baseline on train
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

t0 = time.time()
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(1, 4), min_df=2, max_features=400_000, sublinear_tf=True)
X_train = vec.fit_transform(train.prompt)
clf = LogisticRegression(C=4.0, max_iter=2000)
clf.fit(X_train, train.label)
U = list(clf.classes_).index('U')
p_dev = clf.predict_proba(vec.transform(dev.prompt))[:, U]
print(f'⏱️ trained in {time.time() - t0:.0f} s on {X_train.shape[0]:,} prompts, {X_train.shape[1]:,} features')
s, ru, rs = safety(dev.label, np.where(p_dev >= 0.5, 'U', 'S'))
print(f'📊 dev, threshold 0.50: Safety {s:.2f} | R_U {ru:.2f} | R_S {rs:.2f}')

### 🎚️ Pick the threshold

The Safety Score weighs unsafe recall more, so the best cut-off is often below 0.5. We try thresholds on dev and keep the best one.

> ⚠️ A threshold chosen on dev makes the dev score optimistic. Your Development-phase score on dev-test (labels hidden) is the honest number.

In [ ]:
#@title 🎚️ Scan thresholds on dev
import matplotlib.pyplot as plt
ths = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([(t, *safety(dev.label, np.where(p_dev >= t, 'U', 'S'))) for t in ths],
                     columns=['threshold', 'Safety', 'R_U', 'R_S'])
best = curve.loc[curve.Safety.idxmax()]
THRESHOLD = float(best.threshold)
ax = curve.plot(x='threshold', y=['Safety', 'R_U', 'R_S'], figsize=(7, 3.2), color=['#111', '#c62828', '#2e7d32'])
ax.axvline(THRESHOLD, ls='--', color='gray'); ax.set_ylabel('score'); ax.set_title('dev scores by threshold')
plt.show()
print(f'🎚️ best threshold {THRESHOLD:.2f}: Safety {best.Safety:.2f} | R_U {best.R_U:.2f} | R_S {best.R_S:.2f}')

In [ ]:
#@title ✅ Score dev with the official scorer
dev_pred = pd.DataFrame({'id': dev.id, 'label': np.where(p_dev >= THRESHOLD, 'U', 'S')})
official_score(dev_pred, 'dev.csv')

## 🔬 4 · What did it get wrong?

Reading mistakes is the fastest way to improve. Below: one unsafe prompt the baseline let through, and one safe prompt it flagged.

In [ ]:
#@title 🔬 Show two dev mistakes
wrong = dev.assign(pred=dev_pred.label, p=p_dev)
for gold, name, color in [('U', 'missed attack (U labeled S)', '#c62828'), ('S', 'false alarm (S labeled U)', '#ef6c00')]:
    rows = wrong[(wrong.label == gold) & (wrong.pred != gold)]
    if len(rows):
        ex = rows.sample(1, random_state=1).iloc[0]
        display(HTML(f'''<div style="border-left:5px solid {color};padding:6px 14px;margin:8px 0">
        <div style="font-size:12px;opacity:.7">{ex['id']} · {name} · P(U) = {ex['p']:.2f}</div>
        <div style="white-space:pre-wrap;margin-top:4px">{ex['prompt'][:600]}</div></div>'''))

🔭 **Small model, simple features.** This baseline sees only character patterns: it has no idea what a prompt asks for. Multilingual encoders and LLM-based guards will very likely score higher. Treat it as the starting point to beat.

<details>
<summary>💡 <b>Where to go from here</b></summary>

* 🌐 Fine-tune a multilingual encoder (for example XLM-R or mDeBERTa) on the train set.
* 🤖 Prompt or fine-tune an LLM as a classifier, or adapt an existing guard model.
* 🔤 Normalize the input: back-transliterate to Latin script, or translate to English, before classifying.
* 🎚️ Keep tuning the threshold: unsafe recall counts twice as much as safe recall.
* 📚 Any public data, model or tool is allowed; describe it in your system paper.

</details>

## 📤 5 · Predict dev-test and write the submission

CodaBench takes a **`.zip`** with one file, **`submission.csv`**: a header `id,label`, then one row per prompt with **S** or **U**. The scorer's **format check** catches the problems that make CodaBench reject a file. Run it every time: CodaBench does not show why a file was rejected.

In [ ]:
#@title 📤 Write, check and download submission.zip
p_devtest = clf.predict_proba(vec.transform(devtest.prompt))[:, U]
sub = pd.DataFrame({'id': devtest.id, 'label': np.where(p_devtest >= THRESHOLD, 'U', 'S')})
sub.to_csv('submission.csv', index=False)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('submission.csv')
print(subprocess.run([sys.executable, 'scorer/score.py', '--check', 'submission.zip', 'devtest.csv'],
                     capture_output=True, text=True).stdout)
try:
    from google.colab import files
    files.download('submission.zip')
except ImportError:
    print('📦 submission.zip is in', os.getcwd())

## 🚀 6 · Submit on CodaBench

1. Open [**the CodaBench competition**](https://www.codabench.org/competitions/18479/) and click **Register** (approval is automatic).
2. Go to **My Submissions** and pick the **Development** phase (the **Test** phase opens on Jan 8, 2027).
3. Fill in **Team name** and **Affiliation** (both appear on the leaderboard) and leave **Submit as** on **Yourself**.
4. Upload `submission.zip` and wait for **Finished**. Your Safety Score, $R_U$ and $R_S$ appear on the leaderboard.

Questions go to the **Forum** tab of the competition.

## 🧯 Common errors

| What you see | Why | Fix |
|---|---|---|
| ❌ Status **Failed** on CodaBench | The file broke a format rule | Run the format check cell and read its message |
| `the label must be S or U` | Labels like `0/1` or `safe/unsafe` | Map them to `S` and `U` |
| `exactly one submission.csv` | Wrong file name, or a folder inside the zip | Zip `submission.csv` itself, at the top level |
| A low score with many missing prompts | Predictions for another split | Predict **devtest.csv** in the Development phase |